In [1]:
import os, sys
import re
from datetime import timedelta

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint  # not used in eval, kept for consistency
from tensorflow.keras.optimizers import Adam            # not used in eval, kept for consistency
from tensorflow.keras.losses import MeanAbsoluteError, MeanSquaredError
from scipy.stats import pearsonr

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from astropy.io import fits

try:
    current_folder = os.getcwd()
    hsemetric_folder = os.path.abspath(os.path.join(current_folder, "../HSEMETRIC"))
    sys.path.append(hsemetric_folder)
    import hsemetric
except Exception:
    hsemetric = None
    print("hsemetric not available — Threat_score will be set to NaN.")


# -------------------------
# SpeedNet definition
# -------------------------
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Reshape, LSTM, Dense
from tensorflow.keras.models import Model

def SpeedNet(input_shape, lstm_unit=70, dense_unit=300):
    tf.keras.backend.clear_session()
    np.random.seed(42)
    tf.random.set_seed(42)

    cnn_input = Input(shape=input_shape)
    x = Conv2D(16, (3, 3), activation="relu")(cnn_input)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(32, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    cnn_output = Flatten()(x)

    cnn_output_reshaped = Reshape((1, cnn_output.shape[1]))(cnn_output)
    lstm_output = LSTM(lstm_unit, activation="relu")(cnn_output_reshaped)
    dense_lay = Dense(dense_unit, activation="relu")(lstm_output)
    lstm_output = Dense(1, activation="linear")(dense_lay)

    combined_model = Model(inputs=cnn_input, outputs=lstm_output)
    return combined_model



def prepare_dataset(sw_csv, icme_excel, img_path, std_csv):
    """
    Load and merge solar wind speed, standard deviation, and ICME list.

    Parameters
    ----------
    sw_csv : str
        Path to solar wind CSV file.
    icme_excel : str
        Path to ICME Excel file.
    img_path : str
        Directory containing EUV image files.
    std_csv : str
        Path to CSV containing standard deviation values.

    Returns
    -------
    tuple
        (merged dataframe, dict of date->image_file, icme_dates set)
    """
    df_sw = pd.read_csv(sw_csv)
    df_sw["DATE"] = pd.to_datetime(df_sw["DATE"])
    df_sw = df_sw.rename(columns={"V(Km/s)": "speed"})

    df_std = pd.read_csv(std_csv)
    df_std["DATE"] = pd.to_datetime(df_std["DATE"])
    df_std = df_std.rename(columns={"V(Km/s)": "std_speed"})

    df = pd.merge(df_sw, df_std, on="DATE", how="inner")

    icme_list = pd.read_excel(icme_excel)
    icme_list["Start_time"] = pd.to_datetime(icme_list["Start_time"].str.split(" ").str[0], format="%Y/%m/%d")
    icme_list["End_time"] = pd.to_datetime(icme_list["End_time"].str.split(" ").str[0], format="%Y/%m/%d")

    icme_dates = []
    for start, end in zip(icme_list["Start_time"], icme_list["End_time"]):
        icme_dates.extend(pd.date_range(start, end).tolist())
    icme_dates = set(icme_dates)

    df = df[~df["DATE"].isin(icme_dates)].reset_index(drop=True)

    img_files, img_dates = [], []
    for f in os.listdir(img_path):
        if f.isdigit() and len(f) == 8:
            date = pd.to_datetime(f, format="%Y%m%d")
        else:
            m = re.search(r"(\d{8})", f)
            if not m:
                continue
            date = pd.to_datetime(m.group(1), format="%Y%m%d")

        if date in icme_dates:
            continue
        img_files.append(f)
        img_dates.append(date)

    img_dates, img_files = zip(*sorted(zip(img_dates, img_files)))
    return df, dict(zip(img_dates, img_files)), icme_dates



def match_data(df_sw, img_dict, delay):
    """
    Match solar wind + std to EUV images with forecast delay.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing DATE, speed, std_speed.
    img_dict : dict
        Mapping from date -> image filename.
    delay : int
        Forecast delay (days).

    Returns
    -------
    pd.DataFrame
        Matched dataset with sw_date, img_date, file, speed, std_speed.
    """
    matched = []
    for _, row in df_sw.iterrows():
        sw_date, sw_speed, std_speed = row["DATE"], row["speed"], row["std_speed"]
        img_date = sw_date - timedelta(days=delay)
        if img_date in img_dict:
            matched.append((sw_date, img_date, img_dict[img_date], sw_speed, std_speed))
    return pd.DataFrame(matched, columns=["sw_date", "img_date", "img_file", "speed", "std_speed"])


# -------------------------
# Batch splitting (consecutive by img_date)
# -------------------------
def split_batches(df, batch_size=20):
    batches, i = [], 0
    while i + batch_size <= len(df):
        window = df.iloc[i:i+batch_size]
        if (window["img_date"].diff().dropna() == pd.Timedelta(days=1)).all():
            batches.append(window)
            i += batch_size
        else:
            i += 1
    return batches


# -------------------------
# Random split: Train+Val batches vs Test batches (fixed test for all folds)
# -------------------------
def split_train_val_test(df_matched, test_fraction=0.2, n_splits=5, seed=42, batch_size=20):
    np.random.seed(seed)
    all_batches = split_batches(df_matched, batch_size=batch_size)
    num_batches = len(all_batches)
    n_test = max(1, int(test_fraction * num_batches))
    test_indices = np.random.choice(num_batches, size=n_test, replace=False)

    test_batches = [all_batches[i] for i in test_indices]
    train_val_batches = [b for i, b in enumerate(all_batches) if i not in test_indices]

    print(f"Total batches: {num_batches}, Test batches: {len(test_batches)}, Train+Val batches: {len(train_val_batches)}")
    return train_val_batches, test_batches


# -------------------------
# Load map file (FITS -> np.array, HxW[xC])
# -------------------------
def load_map(filepath):
    with fits.open(filepath) as fit_img:
        arr = np.array(fit_img[0].data, dtype=np.float32)
    if arr.ndim == 2:
        arr = np.expand_dims(arr, -1)
    return arr


# -------------------------
# Metrics helper (and optional HSE metrics)
# -------------------------
def reduced_chi_squared(y_true, y_pred, sigma):
    """
    Calculate the reduced chi-squared (reduced mean square error)
    
    Parameters:
    ----------
    y_true : array-like : True observed values
    y_pred : array-like : Predicted values
    sigma  : array-like : Standard deviation (errors) of the true values
    
    Returns:
    ----------
    float : Reduced chi-squared value

    Examples:
    ---------
    >>> reduced_chi_squared(y_true, 
                           y_pred,
                           sigma, 
        )
    """
    
    if len(y_true) != len(y_pred) or len(y_true) != len(sigma):
        raise ValueError("Input arrays must have the same length")

    # Calculate the reduced chi-squared
    chi_squared = np.sum(((y_true - y_pred) ** 2) / (sigma ** 2))
    reduced_chi_squared = chi_squared / len(y_true)
    
    return reduced_chi_squared

 
def compute_metrics(y_true, y_pred, y_std, hsemetric=None):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mae = MeanAbsoluteError()(y_true, y_pred).numpy()
    mse = MeanSquaredError()(y_true, y_pred).numpy()
    rmse = float(np.sqrt(mse))
    n = len(y_true)

    # --- Uncertainty of MSE
    errors = y_true - y_pred
    sq_errors = errors ** 2
    mse_std = np.std(sq_errors, ddof=1) / np.sqrt(n)

    # --- Uncertainty of RMSE (propagation)
    rmse_std = (0.5 / rmse) * mse_std if rmse > 0 else np.nan

    # guard for tiny arrays
    if len(y_true) > 1 and len(y_pred) > 1:
        corr, p_value = pearsonr(y_true, y_pred)
    else:
        corr, p_value = (np.nan, np.nan)
    
    if abs(corr) < 1 and n > 3:
        sigma_z = 1 / np.sqrt(n - 3)
        corr_std = (1 - corr**2) * sigma_z
    else:
        corr_std = np.nan
        
    # threat score
    
    if hsemetric is not None:
        try:
            tp, fp, fn = hsemetric.BatchwiseHSE(y_true, np.array(y_pred))
            threat_score = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else np.nan
            bias = (tp + fn) / (tp + fp) if (tp + fp) > 0 else np.nan
            ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            far = fp / (tp + fp) if (tp + fp) > 0 else np.nan
        except Exception:
            threat_score = bias = ppv = far = np.nan
    else:
        threat_score = bias = ppv = far = np.nan

    reduced_chi_squ = reduced_chi_squared(y_true, np.array(y_pred), np.array(y_std))

    return dict(MAE=float(mae), MSE=float(mse), MSE_error = float(mse_std), RMSE=float(rmse), 
                RMSE_error = float(rmse_std), Corr=float(corr),
                Corr_std=float(corr_std), P_Value=float(p_value), Threat_Score=float(threat_score),
                Bias=float(bias), PPV=float(ppv), FAR=float(far), Reduced_Chi_Squared = float(reduced_chi_squ))


# -------------------------
# Persistence baselines
# -------------------------
def persistence_27_on_dates(df_sw, sw_dates, period_days=27.0, fill_method=None):
    """
    Classic persistence: predicted(t) = V(t - period_days)
    """
    s = (df_sw[["DATE", "speed"]]
         .dropna()
         .sort_values("DATE")
         .set_index("DATE")["speed"]
         .asfreq("1D"))
    if fill_method in ("ffill", "bfill"):
        s = getattr(s, fill_method)()

    lag = pd.Timedelta(days=period_days)
    preds = []
    for d in sw_dates:
        prev = d - lag
        val = float(s.loc[prev]) if prev in s.index else np.nan
        preds.append(val)
    return np.array(preds, dtype=float)


def persistence_N_on_dates(df_sw, sw_dates, delay, period_days=27.0, fill_method=None):
    """
    N-day persistence WITH delay:
        predicted(t) = V(t + delay - period_days)

    Examples:
        - If delay=4 and N=27: predicted(t) = V(t + 4 - 27).
        - Set period_days to any N you want.
    """
    s = (df_sw[["DATE", "speed"]]
         .dropna()
         .sort_values("DATE")
         .set_index("DATE")["speed"]
         .asfreq("1D"))
    if fill_method in ("ffill", "bfill"):
        s = getattr(s, fill_method)()

    lag = pd.Timedelta(days=period_days)
    preds = []
    for d in sw_dates:
        prev = d + timedelta(days=delay) - lag
        val = float(s.loc[prev]) if prev in s.index else np.nan
        preds.append(val)
    return np.array(preds, dtype=float)


# -------------------------
# Plot: Observed vs Model vs 27-day & N-day Persistence (+ Correlations)
# -------------------------
def plot_results_with_persistence(sw_dates, y_obs, y_pred_model, y_pred_pers27, y_pred_persN,
                                  icme_dates, delay, fold_index,
                                  corr_model, p_model, corr_pers27, p_pers27,
                                  corr_persN, p_persN,
                                  save_path, title_prefix="SpeedNet",
                                  max_nans=5, gap_size=3):
    """
    Four-line time series (Observed, Model, 27-day, N-day) with small markers
    + correlation panel for all three predictions.
    """

    # --- Make DataFrame ---
    df_plot = pd.DataFrame({
        "sw_date": sw_dates,
        "observed": y_obs.reshape(-1),
        "model": y_pred_model.reshape(-1),
        "pers27": y_pred_pers27.reshape(-1),
        "persN": y_pred_persN.reshape(-1)
    }).sort_values("sw_date").reset_index(drop=True)

    # --- Compress long NaN runs ---
    compressed_rows = []
    nan_count = 0
    for _, row in df_plot.iterrows():
        if pd.isna(row["observed"]) and pd.isna(row["model"]) and pd.isna(row["pers27"]) and pd.isna(row["persN"]):
            nan_count += 1
            if nan_count <= max_nans:
                compressed_rows.append(row)
        else:
            nan_count = 0
            compressed_rows.append(row)
    df_plot = pd.DataFrame(compressed_rows).reset_index(drop=True)

    # --- Build artificial index with spacing for gaps ---
    index_with_gaps = []
    current_idx = 0
    prev_date = None
    for d in df_plot["sw_date"]:
        if prev_date is not None and (d - prev_date).days > 1:
            current_idx += gap_size
        index_with_gaps.append(current_idx)
        current_idx += 1
        prev_date = d
    df_plot["plot_idx"] = index_with_gaps

    idx   = df_plot["plot_idx"].values
    dates = df_plot["sw_date"].values
    obs   = df_plot["observed"].values
    pred  = df_plot["model"].values
    pers27 = df_plot["pers27"].values
    persN  = df_plot["persN"].values

    # --- Helper: plot with gaps & small markers ---
    def plot_with_gaps(ax, x, y, color, label, marker, gap_tolerance=gap_size, linewidth=1.0, markersize=3):
        seg_x, seg_y = [], []
        for i in range(len(x)):
            if np.isnan(y[i]):
                if seg_x:
                    ax.plot(seg_x, seg_y, color=color, linewidth=linewidth,
                            marker=marker, markersize=markersize,
                            label=label if label else None)
                    label = None
                    seg_x, seg_y = [], []
                continue
            if seg_x and (x[i] - seg_x[-1]) > gap_tolerance:
                ax.plot(seg_x, seg_y, color=color, linewidth=linewidth,
                        marker=marker, markersize=markersize,
                        label=label if label else None)
                label = None
                seg_x, seg_y = [], []
            seg_x.append(x[i]); seg_y.append(y[i])
        if seg_x:
            ax.plot(seg_x, seg_y, color=color, linewidth=linewidth,
                    marker=marker, markersize=markersize,
                    label=label if label else None)

    # --- Figure ---
    fig = plt.figure(figsize=(32, 8))
    gs = GridSpec(1, 2, figure=fig, width_ratios=[8, 2.8], wspace=0.25)
    ax_ts   = fig.add_subplot(gs[0, 0])
    ax_corr = fig.add_subplot(gs[0, 1])

    plt.style.use("default")
    fig.suptitle(f"{title_prefix}-EUV [Full Cycle], Delay: {delay})",
                 fontsize=16, fontweight="bold", y=1.02)

    # --- Time series (clean colors, thin lines, tiny markers) ---
    plot_with_gaps(ax_ts, idx, pred,   color="blue",   label="SpeedNet-EUV",      marker="o")
    plot_with_gaps(ax_ts, idx, pers27, color="red",    label="27-day",     marker="s")
    plot_with_gaps(ax_ts, idx, persN,  color="orange", label="N-day",      marker="^")
    plot_with_gaps(ax_ts, idx, obs,    color="green",  label="Observed",   marker=".")

    # ICME date tick marks
    for icme_date in icme_dates:
        if np.any(np.abs((dates - np.datetime64(icme_date)).astype("timedelta64[D]").astype(int)) <= 2):
            icme_idx = df_plot.loc[df_plot["sw_date"] == icme_date, "plot_idx"]
            if not icme_idx.empty:
                ax_ts.axvline(x=icme_idx.iloc[0], color="black",
                              linestyle="-", ymax=1, ymin=0.95, linewidth=1.0)

    ax_ts.set_ylabel("V (Km/s)", fontsize=12, fontweight="bold")
    ax_ts.set_xlabel("SW DATE", fontsize=12, fontweight="bold")
    ax_ts.grid(False)
    ax_ts.legend(loc="upper left")

    # Sparse x ticks with readable labels
    if len(dates) > 0:
        step = max(1, len(idx)//15)
        tick_locs = idx[::step]
        tick_labels = [pd.to_datetime(dates[i]).strftime("%d-%m-%Y") for i in range(0, len(dates), step)]
        ax_ts.set_xticks(tick_locs)
        ax_ts.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)

    # --- Correlation panel ---
    mask_model  = ~np.isnan(obs) & ~np.isnan(pred)
    mask_pers27 = ~np.isnan(obs) & ~np.isnan(pers27)
    mask_persN  = ~np.isnan(obs) & ~np.isnan(persN)

    ax_corr.scatter(obs[mask_model],  pred[mask_model],    alpha=0.6, label="Obs vs SpeedNet-EUV",  s=14, c="blue")
    ax_corr.scatter(obs[mask_pers27], pers27[mask_pers27], alpha=0.6, label="Obs vs 27-day", s=14, c="red")
    ax_corr.scatter(obs[mask_persN],  persN[mask_persN],   alpha=0.6, label="Obs vs N-day",  s=14, c="orange")

    # Optional simple fit lines
    if mask_model.sum() > 1:
        m_slope, m_inter = np.polyfit(obs[mask_model], pred[mask_model], 1)
        ax_corr.plot(obs[mask_model], m_slope*obs[mask_model] + m_inter, 'k--', linewidth=1.0, label="SpeedNet-EUV fit")
    if mask_pers27.sum() > 1:
        p27_slope, p27_inter = np.polyfit(obs[mask_pers27], pers27[mask_pers27], 1)
        ax_corr.plot(obs[mask_pers27], p27_slope*obs[mask_pers27] + p27_inter, 'r--', linewidth=1.0, label="27-day fit")
    if mask_persN.sum() > 1:
        pN_slope, pN_inter = np.polyfit(obs[mask_persN], persN[mask_persN], 1)
        ax_corr.plot(obs[mask_persN], pN_slope*obs[mask_persN] + pN_inter, 'orange', linestyle="--", linewidth=1.0, label="N-day fit")

    ax_corr.set_title("Correlation", fontsize=14, fontweight="bold")
    ax_corr.set_xlabel("Observed V (Km/s)", fontsize=12, fontweight="bold")
    ax_corr.set_ylabel("Predicted V (Km/s)", fontsize=12, fontweight="bold")

    ax_corr.text(0.98, 0.04,
                 f"SpeedNet-EUV: r={corr_model:.2f}, p={p_model:.2e}\n"
                 f"27-day: r={corr_pers27:.2f}, p={p_pers27:.2e}\n"
                 f"N-day: r={corr_persN:.2f}, p={p_persN:.2e}",
                 transform=ax_corr.transAxes, fontsize=12,
                 verticalalignment="bottom", horizontalalignment="right",
                 bbox=dict(boxstyle="round", facecolor="white"))

    ax_corr.legend(loc="upper left")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()
    print(f"Saved plot: {save_path}")


# -------------------------
# Main evaluation pipeline (adds N-day persistence with delay)
# -------------------------

def model_train_and_persistence(sw_csv, std_csv, icme_excel, img_path, model_path,
                                        batch_size, lstm_unit, dense_unit,
                                        seed=42, test_fraction=0.2, batch_window=20,
                                        save_path="./results", hsemetric=None,
                                        persistence_period_days=27.0, persistence_fill=None,
                                        N_period_days=27.0):
    """
    1) Builds the SAME test set (from split_train_val_test) for the given delay.
    2) Train the model on X_train and evaluate/test on X_test and compute metrics.
    3) Computes a 27-day persistence baseline on the SAME test dates and metrics.
    4) Plots Observed vs Predicted (Model + Persistence) and dual correlation.
    5) Saves artifacts under save_path and returns two DataFrames:
       - df_metrics_model
       - df_metrics_persistence
    """
    
    
    os.makedirs(save_path, exist_ok=True)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    
    # ---- Prepare data ----
    # df_sw, img_dict, icme_dates = prepare_dataset(sw_csv, icme_excel, img_path)
    df_sw, img_dict, icme_dates = prepare_dataset(sw_csv, icme_excel, img_path, std_csv)
    print(f"Loaded {len(df_sw)} SW records and {len(img_dict)} images from {img_path}")

    delays = [1, 2, 3, 4, 5]  # Which is the average delay
    n_splits = 5
    full_metric_list = []
    

    for delay in delays:
        print(f"\n=== Delay {delay} ===")

        # ---- Match with images based on delay ----
        df_matched = match_data(df_sw, img_dict, delay)
        if df_matched.empty:
            raise ValueError("No matched SW/image records for the specified delay.")

        train_val_batches, test_batches = split_train_val_test(df_matched, test_fraction=test_fraction,
                                                               n_splits=n_splits, seed=seed)

        # n_splits = 5
        num_samples = len(train_val_batches)  # number of train+val batches

        for fold_index in range(n_splits):
            val_start = fold_index * (num_samples // n_splits)
            val_end = val_start + (num_samples // n_splits)
            if val_end > num_samples:
                val_end = num_samples

            # Define train/validation batches
            train_idx = list(range(0, val_start)) + list(range(val_end, num_samples))
            val_idx = list(range(val_start, val_end))

            # Load train/val data
            X_train = np.array([load_map(os.path.join(img_path, r.img_file))
                                for i in train_idx for _, r in train_val_batches[i].iterrows()])
            y_train = np.array([r.speed for i in train_idx for _, r in train_val_batches[i].iterrows()])

            X_val = np.array([load_map(os.path.join(img_path, r.img_file))
                            for i in val_idx for _, r in train_val_batches[i].iterrows()])
            y_val = np.array([r.speed for i in val_idx for _, r in train_val_batches[i].iterrows()])

            # Test set remains fixed
            X_test = np.array([load_map(os.path.join(img_path, r.img_file))
                            for b in test_batches for _, r in b.iterrows()])
            y_test = np.array([r.speed for b in test_batches for _, r in b.iterrows()])
           
            std_test = np.array([r.std_speed for b in test_batches for _, r in b.iterrows()])
            
            sw_test_dates = np.array([r.sw_date for b in test_batches for _, r in b.iterrows()])
            
            print(f"Fold {fold_index}: Train {len(X_train)}, Val {len(X_val)}, Test {len(X_test)}")


            input_shape = X_train.shape[1:]
            
            model = SpeedNet(input_shape, lstm_unit, dense_unit)
            model.compile(optimizer=Adam(1e-4), loss="mse", metrics=["mae", "mse"])

            model_filename = (f"{model_path}/best_model_fold{fold_index}_delay{delay}_"
                            f"bs{batch_size}_lstm{lstm_unit}_dense{dense_unit}.keras")
            
            checkpoint = ModelCheckpoint(model_filename, monitor="val_loss",
                                        save_best_only=True, mode="min", verbose=1)

            history = model.fit(X_train, y_train, epochs=100, batch_size=batch_size,
                                validation_data=(X_val, y_val),
                                callbacks=[checkpoint], verbose=1)

            plt.plot(history.history["loss"], label="train loss")
            plt.plot(history.history["val_loss"], label="val loss")
            plt.xlabel("Epochs"); plt.ylabel("Loss")
            plt.title(f"Fold {fold_index} Delay {delay} BS {batch_size} LSTM {lstm_unit} Dense {dense_unit}")
            plt.legend()
            plt.savefig(model_filename.replace(".keras","_loss.png"))
            plt.close()
            
            
            loaded = load_model(model_filename)
            preds_model = loaded.predict(X_test, verbose=1).reshape(-1)

            # ---- Metrics for the MODEL ----
            metrics_model = compute_metrics(y_test, preds_model, std_test, hsemetric=hsemetric)
            df_metrics_model = pd.DataFrame([{
                "Method": "Model",
                "Fold": fold_index, "Delay": delay, "Batchsize": batch_size,
                "LSTM": lstm_unit, "Dense": dense_unit,
                **metrics_model
            }])
            
            df_prediction_speednet = pd.DataFrame([{
                "Date": sw_test_dates,
                "Observed": y_test,
                "prediction": preds_model,
                "Observed_std": std_test
            }])
            
            speednet_prediction_file = os.path.join(save_path, f"prediction_speednet_fold{fold_index}_delay{delay}.xlsx")
            df_prediction_speednet.to_excel(speednet_prediction_file, index=False)
            
            
            # ---- Classic 27-day Persistence on SAME TEST DATES ----
            preds_pers27 = persistence_27_on_dates(df_sw, sw_test_dates,
                                                period_days=persistence_period_days,
                                                fill_method=persistence_fill)
            df_prediction_p27 = pd.DataFrame([{
                "Date": sw_test_dates,
                "prediction": preds_pers27,
            }])
            p27_prediction_file = os.path.join(save_path, f"prediction_persistence{int(persistence_period_days)}_delay{delay}.xlsx")
            df_prediction_p27.to_excel(p27_prediction_file, index=False)
            
            mask_valid_p27 = ~np.isnan(preds_pers27)
            y_obs_p27 = y_test[mask_valid_p27]
            std_obs_p27 = std_test[mask_valid_p27]
            y_p27     = preds_pers27[mask_valid_p27]
            metrics_p27 = compute_metrics(y_obs_p27, y_p27, std_obs_p27, hsemetric=hsemetric)
            df_metrics_p27 = pd.DataFrame([{
                "Method": f"Persistence{int(persistence_period_days)}",
                "Delay": delay,
                **metrics_p27
            }])
            p27_metrics_file = os.path.join(save_path, f"metrics_persistence{int(persistence_period_days)}_delay{delay}.xlsx")
            df_metrics_p27.to_excel(p27_metrics_file, index=False)
            print(f"Saved {int(persistence_period_days)}-day persistence metrics to {p27_metrics_file}")

            # ---- N-day Persistence WITH delay: predicted(t) = V(t + delay - N) ----
            preds_persN = persistence_N_on_dates(df_sw, sw_test_dates,
                                                delay=delay,
                                                period_days=N_period_days,
                                                fill_method=persistence_fill)
            
            df_prediction_pN = pd.DataFrame([{
                "Date": sw_test_dates,
                "prediction": preds_persN,
            }])
            pN_prediction_file = os.path.join(save_path, f"prediction_persistenceN{int(N_period_days)}_with_delay{delay}.xlsx")
            df_prediction_pN.to_excel(pN_prediction_file, index=False)
            
            mask_valid_pN = ~np.isnan(preds_persN)
            y_obs_pN = y_test[mask_valid_pN]
            std_obs_pN = std_test[mask_valid_pN]
            y_pN     = preds_persN[mask_valid_pN]
            metrics_pN = compute_metrics(y_obs_pN, y_pN, std_obs_pN, hsemetric=hsemetric)
            df_metrics_pN = pd.DataFrame([{
                "Method": f"Nday_with_delay_{int(N_period_days)}",
                "Delay": delay,
                **metrics_pN
            }])
            
            pN_metrics_file = os.path.join(save_path, f"metrics_persistenceN{int(N_period_days)}_with_delay{delay}.xlsx")
            df_metrics_pN.to_excel(pN_metrics_file, index=False)
            print(f"Saved N-day (N={int(N_period_days)}) persistence-with-delay metrics to {pN_metrics_file}")

            # ---- Correlations for annotation (over full / valid masks respectively) ----
            corr_m, p_m = (pearsonr(y_test, preds_model) if len(y_test) > 1 else (np.nan, np.nan))
            corr_p27, p_p27 = (pearsonr(y_obs_p27, y_p27) if len(y_obs_p27) > 1 else (np.nan, np.nan))
            corr_pN,  p_pN  = (pearsonr(y_obs_pN,  y_pN)  if len(y_obs_pN)  > 1 else (np.nan, np.nan))

            # ---- Plot: four lines + correlation panel ----
            plot_file = os.path.join(save_path, f"results_fold{fold_index}_delay{delay}_model_vs_persistence_four_lines.png")
            plot_results_with_persistence(sw_test_dates, y_test, preds_model, preds_pers27, preds_persN,
                                        icme_dates, delay, fold_index,
                                        corr_m, p_m, corr_p27, p_p27, corr_pN, p_pN,
                                        save_path=plot_file,
                                        title_prefix="SpeedNet")
            full_metric_list.append(df_metrics_model)
            full_metric_list.append(df_metrics_p27)
            full_metric_list.append(df_metrics_pN)
            

    # after all folds
    full_df = pd.concat(full_metric_list, ignore_index=True)

    # save once to Excel
    final_metrics_file = os.path.join(save_path, "all_metrics_summary.xlsx")
    full_df.to_excel(final_metrics_file, index=False)
    print(f"Saved combined metrics to {final_metrics_file}")
        


2025-12-09 14:13:23.135507: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


#### SPEEDNET-EUV 
`PARAMS` - `BATCH SIZE = 12`, `LSTM UNITS = 100`, `DENSE UNITS =200`

In [ ]:
SpeedNet_EUV_run_train = model_train_and_persistence(
    sw_csv="../DATASET/Omni_velocity.csv",
    std_csv="../DATASET/standard_deviation_Velocity.csv",
    icme_excel="../DATASET/Original_Full_Richardson_cane_list(2010-2023).xlsx",
    img_path="../DATASET/stonyhurst_three_channel",
    model_path = "../DATASET/Results/SpeedNet_EUV/Full_SC/SpeedNet_EUV_ANALYSIS",
    batch_size=12,            # <- the known hyperparams you obtained
    lstm_unit=100,
    dense_unit=200,
    seed=42,
    test_fraction=0.2,        # same scheme used during training
    batch_window=20,
    save_path = "../DATASET/Results/SpeedNet_EUV/Full_SC/SpeedNet_EUV_ANALYSIS/metrics",
    hsemetric=hsemetric,           # or pass your hsemetric object
    persistence_period_days=27.0,  # classic 27-day baseline: predicted(t)=V(t-27)
    persistence_fill=None,         # or "ffill"/"bfill" if you want daily filling
    N_period_days=27.0             # N-day with delay: predicted(t)=V(t+delay-27). Change N as desired.
)

Loaded 4510 SW records and 3427 images from /home/joseph/DL/Revised_SpeedNet/DATASET/stonyhurst_three_channel

=== Delay 1 ===
Total batches: 90, Test batches: 18, Train+Val batches: 72
Fold 0: Train 1160, Val 280, Test 360


2025-09-02 06:29:57.334817: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-02 06:29:57.335266: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:73:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-02 06:29:57.379312: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:17:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-02 06:29:57.379455: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not open file to read NUMA node: /sys/bus/pci/devices/0000:73:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-09-02 06:29:57.379549: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:961] could not op

Epoch 1/100


2025-09-02 06:30:11.620433: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2025-09-02 06:30:16.382384: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-09-02 06:30:16.533591: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-09-02 06:30:16.533785: W tensorflow/stream_executor/gpu/asm_compiler.cc:80] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2025-09-02 06:30:16.711200: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-09-02 06:30:16.711528: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] INTERNAL: Failed to launch ptxas
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.


97/97 [==============================] - ETA: 0s - loss: 52550.3789 - mae: 173.5264 - mse: 52550.3789
Epoch 1: val_loss improved from inf to 9900.85352, saving model to /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_EUV_ANALYSIS/best_model_fold0_delay1_bs12_lstm100_dense200.keras
97/97 [==============================] - 48s 90ms/step - loss: 52550.3789 - mae: 173.5264 - mse: 52550.3789 - val_loss: 9900.8535 - val_mae: 83.9360 - val_mse: 9900.8535
Epoch 2/100
97/97 [==============================] - ETA: 0s - loss: 8272.6992 - mae: 71.4934 - mse: 8272.6992
Epoch 2: val_loss improved from 9900.85352 to 7118.86084, saving model to /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_EUV_ANALYSIS/best_model_fold0_delay1_bs12_lstm100_dense200.keras
97/97 [==============================] - 6s 61ms/step - loss: 8272.6992 - mae: 71.4934 - mse: 8272.6992 - val_loss: 7118.8608 - val_mae: 62.9677 - val_mse: 7118.8608
Epoch 3/100
96/97 [============================>.] - E

#### SPEEDNET-BM
`PARAMS` - `BATCH SIZE = 12`, `LSTM UNITS = 70`, `DENSE UNITS =200`

In [ ]:
SpeedNet_BM_run_train = model_train_and_persistence(
    sw_csv="../DATASET/Omni_velocity.csv",
    std_csv="../DATASET/standard_deviation_Velocity.csv",
    icme_excel="../DATASET/Original_Full_Richardson_cane_list(2010-2023).xlsx",
    img_path="../DATASET/binary_map",
    model_path = "../DATASET/Results/SpeedNet_BM/Full_SC/SpeedNet_BM_ANALYSIS",
    batch_size=12,            # <- the known hyperparams you obtained
    lstm_unit=100,
    dense_unit=400,
    seed=42,
    test_fraction=0.2,        # same scheme used during training
    batch_window=20,
    save_path = "../DATASET/Results/SpeedNet_BM/Full_SC/SpeedNet_BM_ANALYSIS/metrics",
    hsemetric=hsemetric,           # or pass your hsemetric object
    persistence_period_days=27.0,  # classic 27-day baseline: predicted(t)=V(t-27)
    persistence_fill=None,         # or "ffill"/"bfill" if you want daily filling
    N_period_days=27.0             # N-day with delay: predicted(t)=V(t+delay-27). Change N as desired.
)

Loaded 4510 SW records and 3427 images from /home/joseph/DL/Revised_SpeedNet/DATASET/binary_map

=== Delay 1 ===
Total batches: 90, Test batches: 18, Train+Val batches: 72
Fold 0: Train 1160, Val 280, Test 360
Epoch 1/100
97/97 [==============================] - ETA: 0s - loss: 77737.6953 - mae: 227.9376 - mse: 77737.7109
Epoch 1: val_loss improved from inf to 25186.27539, saving model to /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_BM_ANALYSIS/best_model_fold0_delay1_bs12_lstm100_dense400.keras
97/97 [==============================] - 8s 54ms/step - loss: 77737.6953 - mae: 227.9376 - mse: 77737.7109 - val_loss: 25186.2754 - val_mae: 134.4927 - val_mse: 25186.2754
Epoch 2/100
96/97 [============================>.] - ETA: 0s - loss: 9475.8848 - mae: 76.9231 - mse: 9475.8848
Epoch 2: val_loss improved from 25186.27539 to 19162.94141, saving model to /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_BM_ANALYSIS/best_model_fold0_delay1_bs12_lstm100_dense400.k

96/97 [============================>.] - ETA: 0s - loss: 5522.4463 - mae: 57.4707 - mse: 5522.4463
Epoch 7: val_loss did not improve from 8591.50781
97/97 [==============================] - 5s 54ms/step - loss: 5529.9824 - mae: 57.5047 - mse: 5529.9824 - val_loss: 8657.8965 - val_mae: 72.6986 - val_mse: 8657.8955
Epoch 8/100
96/97 [============================>.] - ETA: 0s - loss: 5355.2251 - mae: 56.7857 - mse: 5355.2251
Epoch 8: val_loss improved from 8591.50781 to 7372.47021, saving model to /home/joseph/DL/Revised_SpeedNet/New/FULL_SC_ANALYSES/SpeedNet_BM_ANALYSIS/best_model_fold0_delay1_bs12_lstm100_dense400.keras
97/97 [==============================] - 6s 58ms/step - loss: 5360.8506 - mae: 56.7280 - mse: 5360.8506 - val_loss: 7372.4702 - val_mae: 66.4669 - val_mse: 7372.4702
Epoch 9/100
96/97 [============================>.] - ETA: 0s - loss: 5336.3291 - mae: 56.4640 - mse: 5336.3291
Epoch 9: val_loss did not improve from 7372.47021
97/97 [==============================] - 5s 56